In [25]:
!pip install torch pandas numpy scikit-learn matplotlib statsmodels gymnasium stable-baselines3 shimmy sb3-contrib --quiet
!pip uninstall -y gym 2>/dev/null
!pip install "stable-baselines3>=2.3.0" "gymnasium>=0.29.0" "shimmy>=1.3.0" --quiet

El sistema no puede encontrar la ruta especificada.


 ## Importaciones

In [26]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList
import torch
import matplotlib.pyplot as plt
import requests
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Imports OK")


FEATURE_COLS = [
    "close",        "volume",       "return_1",     "return_7",
    "rsi_14",       "macd",         "macd_signal",  "bb_position",
    "bb_width",     "atr_14",       "volume_ratio", "hour_sin",
    "hour_cos",     "dow_sin",      "dow_cos",      "fear_greed",
    "return_4h",    "rsi_4h",       "return_1d",    "rsi_1d",
]

N_FEATURES = len(FEATURE_COLS)



SAC_CONFIG = {
    "raw_dir"        : Path.cwd().parent / "data" / "raw",
    "main_dir"       : Path.cwd().parent,
    "data_dir"       : Path.cwd().parent / "data",
    "preprocessed_dir": Path.cwd().parent / "data" / "preprocessed",
    "coins"          : ["BTC", "ETH", "SOL", "AVAX"],
    "granularity"    : "4h",
    "train_ratio"    : 0.70,
    "val_ratio"      : 0.15,
    "feature_cols"   : FEATURE_COLS,
    "horizons"       : [1, 2, 3, 4],
    "horizon_weights": [1.0, 0.85, 0.70, 0.55],
    # Entorno
    "lookback"       : 32,
    "initial_capital": 10_000.0,
    "commission"     : 0.001,
    "slippage"       : 0.0005,
    "max_dd_limit"   : 0.25,
    # Reward shaping
    "reward_alpha"   : 1.0,
    "reward_beta"    : 0.5,
    "reward_gamma"   : 0.3,
    "reward_delta"   : 1.0,
    "reward_epsilon" : 0.1,
    # SAC — FIX: eliminadas claves duplicadas (raw_dir y learning_starts)
    "learning_rate"  : 1e-4,
    "learning_starts": 2_000,
    "buffer_size"    : 100_000,
    "batch_size"     : 256,
    "tau"            : 0.005,
    "gamma"          : 0.99,
    "ent_coef"       : "auto",
    "policy_kwargs"  : dict(log_std_init=-3, net_arch=[256, 256]),
    "net_arch"       : [256, 256],
    # Entrenamiento
    "total_timesteps": 300_000,
    "eval_freq"      : 5_000,
}


Dispositivo: cpu
✅ Imports OK


## Carga y preprocesamiento de datos

In [27]:
def download_fear_greed(limit=2000, verbose=False):
    try:
        resp = requests.get(
            f"https://api.alternative.me/fng/?limit={limit}&format=json", timeout=15)
        resp.raise_for_status()
        data = resp.json()["data"]
        df = pd.DataFrame([
            {"date": pd.to_datetime(int(d["timestamp"]), unit="s", utc=True).normalize(),
             "fear_greed": int(d["value"])}
            for d in data
        ]).sort_values("date").reset_index(drop=True)
        if verbose:
            print(f"  Fear & Greed: {len(df)} días ✅")
        return df
    except Exception as e:
        if verbose:
            print(f"  ⚠ Fear & Greed no disponible ({e}) — usando 50")
        dates = pd.date_range("2020-01-01", periods=2500, freq="D", tz="UTC")
        return pd.DataFrame({"date": dates, "fear_greed": 50})

FG_DF = download_fear_greed(verbose=True)

  Fear & Greed: 2000 días ✅


In [32]:
def calc_indicators(df):
    df = df.copy()
    df["ema_7"]  = df["close"].ewm(span=7,  adjust=False).mean()
    df["ema_14"] = df["close"].ewm(span=14, adjust=False).mean()
    delta = df["close"].diff()
    ag = delta.clip(lower=0).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    al = (-delta).clip(lower=0).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    df["rsi_14"] = 100 - 100 / (1 + ag / (al + 1e-9))
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"]        = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    sma20 = df["close"].rolling(20).mean()
    std20 = df["close"].rolling(20).std()
    bb_u  = sma20 + 2 * std20
    bb_l  = sma20 - 2 * std20
    df["bb_width"]    = (bb_u - bb_l) / (sma20 + 1e-9)
    df["bb_position"] = (df["close"] - bb_l) / (bb_u - bb_l + 1e-9)
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift(1)).abs()
    lc = (df["low"]  - df["close"].shift(1)).abs()
    df["atr_14"] = (pd.concat([hl, hc, lc], axis=1).max(axis=1)
                    .ewm(alpha=1/14, min_periods=14, adjust=False).mean())
    return df

def add_features_1h(df):
    df = df.copy()
    if df["timestamp"].dt.tz is None:
        df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
    df = calc_indicators(df)
    df["volatility_7"] = df["close"].pct_change().rolling(7).std()
    df["return_1"]     = df["close"].pct_change(1)
    df["return_7"]     = df["close"].pct_change(7)
    df["volume_ratio"] = df["volume"] / (df["volume"].rolling(24).mean() + 1e-9)
    df["hour_sin"] = np.sin(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["timestamp"].dt.hour / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 4]:
        df[f"return_lag_{lag}"] = df["return_1"].shift(lag)
    df["momentum_ratio"] = (df["return_1"] / df["return_7"].replace(0, 1e-9)).clip(-10, 10)
    h24 = df["high"].rolling(24).max()
    l24 = df["low"].rolling(24).min()
    df["range_pos_24h"] = (df["close"] - l24) / (h24 - l24 + 1e-9)
    return df

def merge_external(df_1h, fg_df):
    df = df_1h.copy()
    df["date"] = df["timestamp"].dt.normalize()
    df = df.merge(fg_df[["date", "fear_greed"]], on="date", how="left")
    df["fear_greed"] = df["fear_greed"].ffill().fillna(50)
    df.drop(columns=["date"], inplace=True)
    return df

def merge_multitf(df_1h, df_4h_raw, df_1d_raw):
    df_4h = calc_indicators(df_4h_raw.copy())
    df_4h["return_4h"] = df_4h["close"].pct_change(1)
    df_4h_f = df_4h[["timestamp", "rsi_14", "macd", "return_4h"]].copy()
    df_4h_f.columns = ["timestamp", "rsi_4h", "macd_4h", "return_4h"]
    df_1d = calc_indicators(df_1d_raw.copy())
    df_1d["return_1d"] = df_1d["close"].pct_change(1)
    df_1d_f = df_1d[["timestamp", "rsi_14", "return_1d"]].copy()
    df_1d_f.columns = ["timestamp", "rsi_1d", "return_1d"]
    df = pd.merge_asof(df_1h.sort_values("timestamp"),
                       df_4h_f.sort_values("timestamp"),
                       on="timestamp", direction="backward")
    df = pd.merge_asof(df, df_1d_f.sort_values("timestamp"),
                       on="timestamp", direction="backward")
    return df

def add_targets(df, horizons):
    for h in horizons:
        df[f"target_ret_{h}"] = (df["close"].shift(-h) - df["close"]) / df["close"]
    return df

def build_all_datasets(cfg, fg_df, force=False):

    raw_dir = Path(cfg["raw_dir"])
    out_dir = Path(cfg["preprocessed_dir"])

    for coin in cfg["coins"]:

        out = out_dir / f"{coin}_hourly.csv"

        if out.exists() and not force:
            print(f"{coin}: features ya existen ✅")
            continue

        print(f"{coin}: construyendo features...", end=" ")

        p1h = raw_dir / f"{coin}_1h_raw.csv"

        df_1h = (
            pd.read_csv(p1h, parse_dates=["timestamp"])
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        if df_1h["timestamp"].dt.tz is None:
            df_1h["timestamp"] = df_1h["timestamp"].dt.tz_localize("UTC")

        df = add_features_1h(df_1h)

        df = merge_external(df, fg_df)

        p4h = raw_dir / f"{coin}_4h_raw.csv"
        p1d = raw_dir / f"{coin}_1d_raw.csv"

        if p4h.exists() and p1d.exists():

            df_4h = pd.read_csv(p4h, parse_dates=["timestamp"])
            df_1d = pd.read_csv(p1d, parse_dates=["timestamp"])

            for d in [df_4h, df_1d]:
                if d["timestamp"].dt.tz is None:
                    d["timestamp"] = d["timestamp"].dt.tz_localize("UTC")

            df = merge_multitf(df, df_4h, df_1d)

        else:

            for col in [
                "rsi_4h",
                "macd_4h",
                "return_4h",
                "rsi_1d",
                "macd_1d",
                "return_1d"
            ]:
                df[col] = 0.0

        df = add_targets(df, cfg["horizons"])

        df = df.dropna().reset_index(drop=True)

        df.to_csv(out, index=False)

        print(f"{len(df):,} filas ✅")

    for coin in cfg["coins"]:
        out = f"{cfg['data_dir']}/{coin}_hourly.csv"
        if os.path.exists(out) and not force:
            print(f"  {coin}: features ya existen ✅")
            continue
        print(f"  {coin}: construyendo features...", end=" ")
        p1h = f"{cfg['raw_dir']}/{coin}_1h_raw.csv"
        df_1h = (pd.read_csv(p1h, parse_dates=["timestamp"])
                   .sort_values("timestamp").reset_index(drop=True))
        if df_1h["timestamp"].dt.tz is None:
            df_1h["timestamp"] = df_1h["timestamp"].dt.tz_localize("UTC")
        df = add_features_1h(df_1h)
        df = merge_external(df, fg_df)
        p4h = f"{cfg['raw_dir']}/{coin}_4h_raw.csv"
        p1d = f"{cfg['raw_dir']}/{coin}_1d_raw.csv"
        if os.path.exists(p4h) and os.path.exists(p1d):
            df_4h = pd.read_csv(p4h, parse_dates=["timestamp"])
            df_1d = pd.read_csv(p1d, parse_dates=["timestamp"])
            for d in [df_4h, df_1d]:
                if d["timestamp"].dt.tz is None:
                    d["timestamp"] = d["timestamp"].dt.tz_localize("UTC")
            df = merge_multitf(df, df_4h, df_1d)
        else:
            for col in ["rsi_4h", "macd_4h", "return_4h", "rsi_1d", "macd_1d", "return_1d"]:
                df[col] = 0.0
        df = add_targets(df, cfg["horizons"])
        df = df.dropna().reset_index(drop=True)
        df.to_csv(out, index=False)
        print(f"{len(df):,} filas ✅")

build_all_datasets(SAC_CONFIG, FG_DF)

def load_coin(coin):
    path = f"{SAC_CONFIG["preprocessed_dir"]}/{coin}_hourly.csv"
    df = pd.read_csv(path, parse_dates=["timestamp"])
    print(f"  [{coin}] <- {path}  ({len(df):,} filas)")
    return df

def preprocess(df, coin):
    if "fear_greed" not in df.columns:
        print(f"  [{coin}] ⚠️  fear_greed ausente → imputando 50")
        df["fear_greed"] = 50.0
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"[{coin}] Features faltantes: {missing}")
    before = len(df)
    df = df.dropna(subset=FEATURE_COLS + ["high", "low"])
    if (dropped := before - len(df)) > 0:
        print(f"  [{coin}] Eliminadas {dropped} filas con NaN")
    no_price = [c for c in FEATURE_COLS if c not in ["open", "high", "low", "close"]]
    for col in no_price:
        mu, sigma = df[col].mean(), df[col].std() + 1e-8
        z = (df[col] - mu) / sigma
        df[col] = df[col].where(z.abs() < 10, df[col].median())
    if isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index().rename(columns={"index": "timestamp"})
    else:
        df = df.reset_index(drop=True)
    return df

def temporal_split(df, train_ratio=0.70, val_ratio=0.15):
    n       = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    return (df.iloc[:n_train].reset_index(drop=True),
            df.iloc[n_train:n_train + n_val].reset_index(drop=True),
            df.iloc[n_train + n_val:].reset_index(drop=True))

BTC: features ya existen ✅
ETH: features ya existen ✅
SOL: features ya existen ✅
AVAX: features ya existen ✅
  BTC: features ya existen ✅
  ETH: features ya existen ✅
  SOL: features ya existen ✅
  AVAX: features ya existen ✅


In [33]:
print("=" * 60)
print("  CELDA 3 · Carga de datos")
print("=" * 60)
datasets = {}
for coin in SAC_CONFIG["coins"]:
    print(f"\n▶ {coin}")
    df_raw  = load_coin(coin)
    df_proc = preprocess(df_raw, coin)
    df_tr, df_va, df_te = temporal_split(
        df_proc, SAC_CONFIG["train_ratio"], SAC_CONFIG["val_ratio"])
    datasets[coin] = {"train": df_tr, "val": df_va, "test": df_te}
    ts = "timestamp" if "timestamp" in df_proc.columns else df_proc.columns[0]
    print(f"  Train : {len(df_tr):>5} velas  "
          f"({df_tr[ts].iloc[0].date()} → {df_tr[ts].iloc[-1].date()})")
    print(f"  Val   : {len(df_va):>5} velas  "
          f"({df_va[ts].iloc[0].date()} → {df_va[ts].iloc[-1].date()})")
    print(f"  Test  : {len(df_te):>5} velas  "
          f"({df_te[ts].iloc[0].date()} → {df_te[ts].iloc[-1].date()})")
print(f"\n✅ Datos listos para {len(datasets)} coins")

  CELDA 3 · Carga de datos

▶ BTC
  [BTC] <- c:\Users\vicen\Documents\TFM-NeuroQuant\data\preprocessed/BTC_hourly.csv  (55,240 filas)
  Train : 38668 velas  (2020-01-15 → 2024-06-14)
  Val   :  8286 velas  (2024-06-14 → 2025-05-25)
  Test  :  8286 velas  (2025-05-25 → 2026-05-06)

▶ ETH
  [ETH] <- c:\Users\vicen\Documents\TFM-NeuroQuant\data\preprocessed/ETH_hourly.csv  (55,240 filas)
  Train : 38668 velas  (2020-01-15 → 2024-06-14)
  Val   :  8286 velas  (2024-06-14 → 2025-05-25)
  Test  :  8286 velas  (2025-05-25 → 2026-05-06)

▶ SOL
  [SOL] <- c:\Users\vicen\Documents\TFM-NeuroQuant\data\preprocessed/SOL_hourly.csv  (48,988 filas)
  Train : 34291 velas  (2020-10-02 → 2024-08-31)
  Val   :  7348 velas  (2024-08-31 → 2025-07-03)
  Test  :  7349 velas  (2025-07-03 → 2026-05-06)

▶ AVAX
  [AVAX] <- c:\Users\vicen\Documents\TFM-NeuroQuant\data\preprocessed/AVAX_hourly.csv  (38,783 filas)
  Train : 27148 velas  (2021-12-02 → 2025-01-06)
  Val   :  5817 velas  (2025-01-06 → 2025-09-05)
  T

## Entorno Gymnasium

In [34]:
class CryptoTradingEnv(gym.Env):
    """
    Entorno de trading para SAC.

    CORRECCIONES aplicadas:
    - FIX CRÍTICO: self.units como variable de estado en _reset_internals.
      El cálculo anterior (prev_pv - capital) / entry crecía
      exponencialmente en cada step de MANTENER → overflow ($26 cuatrillones).
    - _obs_buf preallocado: evita np.concatenate + astype en cada step.
    """
    metadata  = {"render_modes": ["human"]}
    SELL_TH   = -0.33
    BUY_TH    =  0.33
    ACT_NAMES = {0: "VENDER", 1: "MANTENER", 2: "COMPRAR"}

    def __init__(
        self,
        df, feat_mean, feat_std,
        lookback=32, initial_capital=10_000.0,
        commission=0.001, slippage=0.0005, max_dd_limit=0.25,
        reward_alpha=1.0, reward_beta=0.5, reward_gamma=0.3,
        reward_delta=1.0, reward_epsilon=0.1, render_mode=None,
    ):
        super().__init__()
        self.df          = df.reset_index(drop=True)
        self.lb          = lookback
        self.cap0        = initial_capital
        self.commission  = commission
        self.slippage    = slippage
        self.max_dd      = max_dd_limit
        self.render_mode = render_mode
        self.α = reward_alpha
        self.β = reward_beta
        self.γ = reward_gamma
        self.δ = reward_delta
        self.ε = reward_epsilon

        raw        = df[FEATURE_COLS].values.astype(np.float32)
        normed     = (raw - feat_mean) / feat_std
        self._feat = np.clip(
            np.nan_to_num(normed, nan=0., posinf=3., neginf=-3.), -5., 5.)

        n_obs         = lookback * N_FEATURES + 3
        self._obs_buf = np.zeros(n_obs, dtype=np.float32)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_obs,), dtype=np.float32)
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

        self._reset_internals()

    def _reset_internals(self):
        self.step_    = self.lb
        self.capital  = self.cap0
        self.position = 0
        self.entry    = 0.0
        self.units    = 0.0
        self.pv       = self.cap0
        self.peak     = self.cap0
        self.rets     = []
        self.acts     = []
        self.trades   = []
        self.pv_hist  = [self.cap0]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_internals()
        return self._get_obs(), {}

    def _get_obs(self):
        n = self.lb * N_FEATURES
        self._obs_buf[:n]  = self._feat[self.step_ - self.lb: self.step_].ravel()
        self._obs_buf[n]   = float(self.position)
        self._obs_buf[n+1] = (self.pv - self.cap0) / self.cap0
        self._obs_buf[n+2] = self.capital / self.cap0
        return self._obs_buf.copy()

    def _decode(self, raw):
        if   raw < self.SELL_TH: return 0, abs(raw)
        elif raw > self.BUY_TH:  return 2, abs(raw)
        else:                    return 1, 0.0

    def step(self, action):
        raw          = float(action[0])
        signal, size = self._decode(raw)
        price        = float(self.df.loc[self.step_, "close"])
        prev_pv      = self.pv
        tc           = 0.0

        self.acts.append(signal)

        # ── COMPRAR ──────────────────────────────────────────────────────
        if signal == 2 and self.position == 0:
            eff           = price * (1 + self.slippage)
            inv           = self.capital * min(size, 1.0)
            tc            = inv * self.commission
            self.capital -= inv + tc
            self.units    = inv / eff    # ← ¿tienes esta línea?
            self.position = 1
            self.entry    = eff

        # ── VENDER ───────────────────────────────────────────────────────
        elif signal == 0 and self.position == 1:
            eff           = price * (1 - self.slippage)
            proceeds      = self.units * eff
            tc            = proceeds * self.commission
            self.capital += proceeds - tc
            self.units    = 0.0
            self.position = 0
            self.entry    = 0.0


        # ── VALORAR PORTFOLIO ─────────────────────────────────────────────
        if self.position == 1:
            self.pv = self.capital + self.units * price   # ← ¿y esta?
        else:
            self.pv = self.capital

        step_ret = (self.pv - prev_pv) / (prev_pv + 1e-8)
        self.rets.append(step_ret)
        self.pv_hist.append(self.pv)
        self.peak = max(self.peak, self.pv)
        dd        = (self.peak - self.pv) / (self.peak + 1e-8)
        reward    = self._compute_reward(step_ret, tc)

        self.step_ += 1
        terminated  = self.step_ >= len(self.df) - 1
        truncated   = dd > self.max_dd
        if truncated:
            reward -= 5.0
        reward = float(np.clip(reward, -10., 10.))

        return self._get_obs(), reward, terminated, truncated, {
            "portfolio_value": self.pv,
            "drawdown"       : dd,
            "position"       : self.position,
            "action_name"    : self.ACT_NAMES[signal],
            "n_trades"       : len(self.trades),
            "step_return"    : step_ret,
        }

    def _compute_reward(self, step_ret, tc):
        PPA = 1512
        r   = np.array(self.rets[-24:])
        rew = 0.0
        if len(r) >= 2:
            mu       = r.mean()
            sigma    = r.std() + 1e-8
            rew     += self.α * (mu / sigma) * np.sqrt(PPA)
            neg      = r[r < 0]
            downside = neg.std() + 1e-8 if len(neg) > 0 else 1e-8
            rew     += self.β * (mu / downside) * np.sqrt(PPA)
        dd   = (self.peak - self.pv) / (self.peak + 1e-8)
        rew -= self.γ * (dd ** 2)
        rew -= self.δ * tc / (self.pv + 1e-8)
        if len(self.acts) >= 3:
            rc    = self.acts[-3:]
            flips = sum(1 for i in range(1, 3) if rc[i] != rc[i - 1])
            rew  -= self.ε * flips
        return float(rew)

    def render(self):
        if self.render_mode == "human":
            price = float(self.df.loc[self.step_ - 1, "close"])
            print(f"  Step {self.step_:4d} | ${price:>10,.2f} | "
                  f"Portfolio ${self.pv:>10,.2f}")

print("✅ CryptoTradingEnv definido")
print(f"   Obs shape : ({SAC_CONFIG['lookback']} × {N_FEATURES} + 3)"
      f" = {SAC_CONFIG['lookback'] * N_FEATURES + 3} dims")

✅ CryptoTradingEnv definido
   Obs shape : (32 × 20 + 3) = 643 dims


## Construcción de entornos

In [35]:
def compute_train_stats(df_train):
    feat = df_train[FEATURE_COLS].values.astype(np.float32)
    return np.nanmean(feat, axis=0), np.nanstd(feat, axis=0) + 1e-8

def make_env_fn(df, mean_, std_, cfg, monitor=True):
    def _init():
        env = CryptoTradingEnv(
            df=df, feat_mean=mean_, feat_std=std_,
            lookback=cfg["lookback"],
            initial_capital=cfg["initial_capital"],
            commission=cfg["commission"],
            slippage=cfg["slippage"],
            max_dd_limit=cfg["max_dd_limit"],
            reward_alpha=cfg["reward_alpha"],
            reward_beta=cfg["reward_beta"],
            reward_gamma=cfg["reward_gamma"],
            reward_delta=cfg["reward_delta"],
            reward_epsilon=cfg["reward_epsilon"],
        )
        return Monitor(env) if monitor else env
    return _init

In [36]:
print("=" * 60)
print("  CELDA 5 · Creación de entornos")
print("=" * 60)
envs       = {}
coin_stats = {}
for coin in SAC_CONFIG["coins"]:
    mean_, std_ = compute_train_stats(datasets[coin]["train"])
    coin_stats[coin] = (mean_, std_)

    train_vec = DummyVecEnv([
        make_env_fn(datasets[coin]["train"], mean_, std_, SAC_CONFIG, monitor=True)
    ])

    train_vec = VecNormalize(
        train_vec, norm_obs=False, norm_reward=True, clip_reward=10.0)

    val_env  = make_env_fn(
        datasets[coin]["val"],  mean_, std_, SAC_CONFIG, monitor=False)()
    test_env = make_env_fn(
        datasets[coin]["test"], mean_, std_, SAC_CONFIG, monitor=False)()

    envs[coin] = {"train_vec": train_vec, "val": val_env, "test": test_env}

    # Sanity check: obs inicial libre de NaN/Inf
    obs_check = train_vec.reset()
    assert np.isfinite(obs_check).all(), f"[{coin}] NaN/Inf en obs inicial"

    print(f"  [{coin}] ✅  "
          f"train={len(datasets[coin]['train'])} | "
          f"val={len(datasets[coin]['val'])} | "
          f"test={len(datasets[coin]['test'])} velas  · "
          f"obs_dim={train_vec.observation_space.shape[0]}")
print(f"\n✅ {len(envs)} entornos creados")

  CELDA 5 · Creación de entornos
  [BTC] ✅  train=38668 | val=8286 | test=8286 velas  · obs_dim=643
  [ETH] ✅  train=38668 | val=8286 | test=8286 velas  · obs_dim=643
  [SOL] ✅  train=34291 | val=7348 | test=7349 velas  · obs_dim=643
  [AVAX] ✅  train=27148 | val=5817 | test=5818 velas  · obs_dim=643

✅ 4 entornos creados


## Extracción de modelo

In [40]:
trained_models = {}

print(SAC_CONFIG['main_dir'] / "models")
for coin in SAC_CONFIG["coins"]:
    best_path = f"${SAC_CONFIG['main_dir']}/models/sac_{coin}_best.zip"
    print(best_path)
    if Path(best_path).exists():
        trained_models[coin] = SAC.load(
            best_path,
            env    = envs[coin]["train_vec"],
            device = "auto",
        )
        print(f"  [{coin}] ✅ cargado desde {best_path}")
    else:
        print(f"  [{coin}] ⚠️  no se ha encontrado")

c:\Users\vicen\Documents\TFM-NeuroQuant\models
$c:\Users\vicen\Documents\TFM-NeuroQuant/models/sac_BTC_best.zip
  [BTC] ⚠️  no se ha encontrado
$c:\Users\vicen\Documents\TFM-NeuroQuant/models/sac_ETH_best.zip
  [ETH] ⚠️  no se ha encontrado
$c:\Users\vicen\Documents\TFM-NeuroQuant/models/sac_SOL_best.zip
  [SOL] ⚠️  no se ha encontrado
$c:\Users\vicen\Documents\TFM-NeuroQuant/models/sac_AVAX_best.zip
  [AVAX] ⚠️  no se ha encontrado


## Métricas en test

In [ ]:
results = {}
for coin in SAC_CONFIG["coins"]:
    print(f"\n▶ [{coin}]")
    sac_r = run_episode(trained_models[coin], envs[coin]["test"], deterministic=True)
    bh_r  = buy_and_hold(datasets[coin]["test"], SAC_CONFIG["initial_capital"])
    results[coin] = {"sac": sac_r, "bh": bh_r}

    ms, mb = sac_r["metrics"], bh_r["metrics"]
    print(f"  Retorno  SAC / B&H : {ms['total_return_pct']:>7.2f}% / {mb['total_return_pct']:>7.2f}%")
    print(f"  Sharpe   SAC / B&H : {ms['sharpe_ratio']:>7.4f} / {mb['sharpe_ratio']:>7.4f}")
    print(f"  Sortino  SAC / B&H : {ms['sortino_ratio']:>7.4f} / {mb['sortino_ratio']:>7.4f}")
    print(f"  Max DD   SAC / B&H : {ms['max_drawdown_pct']:>7.2f}% / {mb['max_drawdown_pct']:>7.2f}%")
    print(f"  Win rate           : {ms['win_rate_pct']:>7.2f}%")
    print(f"  Trades             : {len(sac_r['trades'])}")

## Quity curve vs Buy & Hold

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, coin in enumerate(SAC_CONFIG["coins"]):
    ax  = axes[i]
    sac = results[coin]["sac"]["portfolio"]
    bh  = results[coin]["bh"]["portfolio"]

    ax.plot(sac, label="SAC Agent", color="#2563EB", linewidth=1.5)
    ax.plot(bh,  label="Buy & Hold", color="#DC2626", linewidth=1.5, linestyle="--")
    ax.axhline(SAC_CONFIG["initial_capital"], color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(f"{coin}  |  Sharpe SAC: {results[coin]['sac']['metrics']['sharpe_ratio']:.3f}")
    ax.set_ylabel("Portfolio Value ($)")
    ax.set_xlabel("Steps")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("SAC Agent vs Buy & Hold · Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/results/equity_curves.png", dpi=150)
plt.show()
print("✅ Gráfica guardada → results/equity_curves.png")